In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder, LabelBinarizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction import DictVectorizer
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..', 'ml-app')))
from preprocessing import WindowingMethod, FeatureExtractor

In [3]:
random_state = 42
data_path = os.path.join('..', 'data')
model_path = os.path.join('data', 'models')

In [ ]:
classifiers = {
    "Logistic Regression": (
        "logreg",
        LogisticRegression(random_state=random_state),
        {
            "classifier__solver": ["liblinear", "lbfgs", "saga", "newton-cg"],
            "classifier__l1_ratio": [0, 0.1, 0.3, 0.5, 0.7, 0.9, 1],
            "classifier__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "classifier__class_weight": [None, "balanced"],
            "classifier__max_iter": [1000, 2000],
        },
    ),
    "Random Forest": (
        "rf",
        RandomForestClassifier(random_state=random_state),
        {
            "classifier__n_estimators": [100, 200, 300, 500, 800],
            "classifier__max_depth": [None, 4, 6, 8, 12, 16, 24],
            "classifier__min_samples_split": [2, 5, 10, 20],
            "classifier__min_samples_leaf": [1, 2, 4, 8],
            "classifier__max_features": ["sqrt", "log2", 0.4, 0.6, 0.8, None],
            "classifier__bootstrap": [True],
            "classifier__class_weight": [None, "balanced", "balanced_subsample"],
            "classifier__criterion": ["gini", "entropy", "log_loss"],
        },
    ),
    "MLP Classifier": (
        "mlp",
        MLPClassifier(random_state=random_state),
        {
            "classifier__hidden_layer_sizes": [
                (8,),
                (16,),
                (32,),
                (64,),
                (32, 16),
                (64, 32),
                (64, 32, 16),
                (128, 64),
            ],
            "classifier__activation": ["relu", "tanh"],
            "classifier__alpha": [1e-5, 1e-4, 1e-3, 1e-2],
            "classifier__learning_rate_init": [1e-4, 3e-4, 1e-3, 3e-3],
            "classifier__batch_size": [16, 32, 64, 128],
            "classifier__max_iter": [500, 1000, 2000],
            "classifier__early_stopping": [True],
            "classifier__validation_fraction": [0.1, 0.15, 0.2],
        },
    ),
}

In [ ]:
def create_pipeline(classifier_name: str) -> Pipeline:
    if classifier_name not in classifiers:
        raise ValueError(
            f"Classifier '{classifier_name}' not found. Available classifiers: {list(classifiers.keys())}"
        )

    classifier_key, classifier, _ = classifiers[classifier_name]

    pipeline = Pipeline(
        [
            (
                "windowing",
                WindowingMethod(method=FeatureExtractor(), window_size=10, stride=2),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "classifier", 
                classifier
            ),
        ]
    )

    return pipeline